In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd

# === 路径配置 ===
PATH = "/Users/liuliangcheng/Desktop/Duke/capstone/temperature"
OUT = os.path.join(PATH, "county_heat_index_2000_2020.csv")

# NOAA 历史长度（从文件标题看是 131 年）
HIST_YEARS = 131


def load_one(csv_path: str) -> pd.DataFrame:
    """
    读取单个 NOAA 县级夏季最高温 CSV，清洗列并计算 heat_index（方案C）
    """
    # 从文件名抓年份
    m = re.search(r"(\d{4})\.csv$", os.path.basename(csv_path))
    year = int(m.group(1)) if m else None

    # 跳过以 '#' 开头的注释行
    df = pd.read_csv(csv_path, sep=",", comment="#", engine="python")
    # 去掉列名首尾空格
    df.columns = df.columns.str.strip()

    # 列名规范化一个映射，兼容可能的大小写/空格
    rename_map = {
        "ID": "id",
        "Name": "name",
        "State": "state",
        "Value": "value",
        "Anomaly": "anomaly",
        "Rank": "rank",
        "1901-2000 Mean": "mean_1901_2000",
    }
    # 只重命名存在的
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    # 清洗 Rank（去掉 * 号）
    if "rank" in df.columns:
        df["rank"] = pd.to_numeric(
            df["rank"].astype(str).str.replace("*", "", regex=False), errors="coerce"
        )

    # 数值化
    for col in ["value", "anomaly", "mean_1901_2000"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 添加年份
    df["year"] = year

    # 计算方案 C：0.6 * 相对偏离 + 0.4 * 归一化排名
    # 相对偏离：(value - mean) / mean
    rel_dev = (df["value"] - df["mean_1901_2000"]) / df["mean_1901_2000"]
    rank_norm = df["rank"] / HIST_YEARS
    df["heat_index"] = 0.6 * rel_dev + 0.4 * rank_norm

    # 只输出需要的字段（也保留原始度量，便于后续调参/核对）
    keep_cols = [
        "id",
        "name",
        "state",
        "year",
        "value",
        "anomaly",
        "rank",
        "mean_1901_2000",
        "heat_index",
    ]
    df = df[[c for c in keep_cols if c in df.columns]].copy()

    return df


# === 批量读取 2000–2020 ===
files = sorted(glob.glob(os.path.join(PATH, "*.csv")))
# 只要 2000..2020 年的
files = [
    f
    for f in files
    if re.search(r"(200[0-9]|201[0-9]|2020)\.csv$", os.path.basename(f))
]

all_parts = []
for f in files:
    try:
        part = load_one(f)
        all_parts.append(part)
        print(f"✅ {os.path.basename(f)} -> {part.shape}")
    except Exception as e:
        print(f"❌ 读取失败 {os.path.basename(f)}: {e}")

all_df = pd.concat(all_parts, ignore_index=True)

# 简单排序 & 基本检查
all_df = all_df.sort_values(["state", "name", "year"]).reset_index(drop=True)
print("合并完成：", all_df.shape)
print(all_df.head(10))

# 保存
all_df.to_csv(OUT, index=False)
print(f"已保存到：{OUT}")

✅ 2000.csv -> (3107, 8)
✅ 2001.csv -> (3107, 8)
✅ 2002.csv -> (3107, 8)
✅ 2003.csv -> (3107, 8)
✅ 2004.csv -> (3107, 8)
✅ 2005.csv -> (3107, 8)
✅ 2006.csv -> (3107, 8)
✅ 2007.csv -> (3107, 8)
✅ 2008.csv -> (3107, 8)
✅ 2009.csv -> (3107, 8)
✅ 2010.csv -> (3107, 8)
✅ 2011.csv -> (3107, 8)
✅ 2012.csv -> (3107, 8)
✅ 2013.csv -> (3107, 8)
✅ 2014.csv -> (3107, 8)
✅ 2015.csv -> (3107, 8)
✅ 2016.csv -> (3107, 8)
✅ 2017.csv -> (3107, 8)
✅ 2018.csv -> (3107, 8)
✅ 2019.csv -> (3107, 8)
✅ 2020.csv -> (3107, 8)
合并完成： (65247, 8)
       id            name    state  year  value   rank  mean_1901_2000  \
0  AL-001  Autauga County  Alabama  2000   93.8  125.0            90.9   
1  AL-001  Autauga County  Alabama  2001   88.9   18.0            90.9   
2  AL-001  Autauga County  Alabama  2002   90.8   60.0            90.9   
3  AL-001  Autauga County  Alabama  2003   88.0    3.0            90.9   
4  AL-001  Autauga County  Alabama  2004   88.3   11.0            90.9   
5  AL-001  Autauga County  Alabama 

In [ ]:
import pandas as pd

# 读取你整理好的文件
path = "/Users/liuliangcheng/Desktop/Duke/capstone/temperature/county_heat_index_2000_2020.csv"

df = pd.read_csv(path)

# 随机展示 10 行数据
print(df.sample(10, random_state=42))

           id                 name           state  year  value   rank  \
53205  TX-097         Cooke County           Texas  2012   95.4  107.0   
31061  MO-069       Dunklin County        Missouri  2002   89.7   61.0   
48311  SC-045    Greenville County  South Carolina  2011   90.2  129.0   
55236  TX-291       Liberty County           Texas  2006   91.8   77.0   
38461  NY-083    Rensselaer County        New York  2010   80.6  118.0   
4148   CA-081     San Mateo County      California  2011   71.0   22.0   
14277  IN-031       Decatur County         Indiana  2018   82.8   38.0   
39881  NC-095          Hyde County  North Carolina  2002   88.0  114.0   
32428  MO-187  St. Francois County        Missouri  2004   82.2    2.0   
14858  IN-091       LaPorte County         Indiana  2011   82.7   89.0   

       mean_1901_2000  heat_index  
53205            93.4    0.339566  
31061            90.1    0.183596  
48311            86.2    0.421735  
55236            91.5    0.237082  
38461

In [ ]:
import pandas as pd
import numpy as np

path = "/Users/liuliangcheng/Desktop/Duke/capstone/temperature/county_heat_index_2000_2020.csv"
df = pd.read_csv(path)

# --- 相对异常（你已有，重算确保存在） ---
df["rel_dev"] = (df["value"] - df["mean_1901_2000"]) / df["mean_1901_2000"]

# --- 跨地区绝对热度（按年标准化/百分位） ---
df["abs_z_year"] = df.groupby("year")["value"].transform(
    lambda s: (s - s.mean()) / s.std(ddof=0)
)
df["abs_pct_year"] = df.groupby("year")["value"].transform(lambda s: s.rank(pct=True))

# --- 极端高温超额强度 ---
df["exceed_90"] = (df["value"] - 90).clip(lower=0)
df["exceed_95"] = (df["value"] - 95).clip(lower=0)
df["exceed_100"] = (df["value"] - 100).clip(lower=0)


# --- 归一化到 0-1（稳健起见，可按 1%/99% 分位限幅后再缩放） ---
def minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else 0.0


df["_n_abs_pct_year"] = df.groupby("year")["abs_pct_year"].transform(
    minmax
)  # 实际与自身相同，但保持接口一致
df["_n_rel_dev"] = df.groupby("year")["rel_dev"].transform(minmax)  # 也可用全局 minmax
df["_n_ex95"] = df.groupby("year")["exceed_95"].transform(minmax)

# --- 综合指数（可调权重） ---
df["HSI"] = (
    0.35 * df["_n_abs_pct_year"] + 0.35 * df["_n_rel_dev"] + 0.30 * df["_n_ex95"]
)

# 清理临时列（可选）
df = df.drop(columns=["_n_abs_pct_year", "_n_rel_dev", "_n_ex95"])

# 保存新文件
out = path.replace(".csv", "_with_HSI.csv")
df.to_csv(out, index=False)
print("Saved:", out)
print(
    df[
        [
            "id",
            "name",
            "state",
            "year",
            "value",
            "rel_dev",
            "abs_pct_year",
            "exceed_95",
            "HSI",
        ]
    ].sample(10, random_state=7)
)

Saved: /Users/liuliangcheng/Desktop/Duke/capstone/temperature/county_heat_index_2000_2020_with_HSI.csv
           id              name           state  year  value   rel_dev  \
30551  MO-019      Boone County        Missouri  2017   85.2 -0.018433   
24900  MA-013    Hampden County   Massachusetts  2015   79.8  0.002513   
48485  SC-061        Lee County  South Carolina  2017   89.4  0.000000   
28400  MN-153       Todd County       Minnesota  2008   77.5 -0.003856   
5073   CO-051   Gunnison County        Colorado  2012   74.4  0.059829   
30383  MO-003     Andrew County        Missouri  2017   85.3 -0.008140   
47477  PA-109     Snyder County    Pennsylvania  2017   80.6 -0.006165   
46143  OR-055    Sherman County          Oregon  2006   83.6  0.037221   
63904  WI-071  Manitowoc County       Wisconsin  2001   78.3  0.020860   
2498   AR-073  Lafayette County        Arkansas  2020   89.1 -0.033623   

       abs_pct_year  exceed_95       HSI  
30551      0.478730        0.0  0.24649

In [ ]:
# ====== 0. Setup ======
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "/Users/liuliangcheng/Desktop/Duke/capstone/temperature"
# 优先使用带 HSI 的文件；没有就用基础文件并现场补算
CANDIDATES = [
    os.path.join(DATA_PATH, "county_heat_index_2000_2020_with_HSI.csv"),
    os.path.join(DATA_PATH, "county_heat_index_2000_2020.csv"),
]
OUT_DIR = os.path.join(DATA_PATH, "figs")
os.makedirs(OUT_DIR, exist_ok=True)


def load_data():
    for p in CANDIDATES:
        if os.path.exists(p):
            df = pd.read_csv(p)
            print("Loaded:", p)
            return df, p
    raise FileNotFoundError(
        "找不到整理后的 CSV。请先生成 county_heat_index_2000_2020.csv"
    )


df, used_path = load_data()

# 兜底：若没有 HSI，则根据你之前的方案补齐用于建模的三个维度 + HSI
need_hsi = "HSI" not in df.columns
if need_hsi:
    print("HSI not found. Calculating HSI on the fly.")
    df["rel_dev"] = (df["value"] - df["mean_1901_2000"]) / df["mean_1901_2000"]
    df["abs_z_year"] = df.groupby("year")["value"].transform(
        lambda s: (s - s.mean()) / s.std(ddof=0)
    )
    df["abs_pct_year"] = df.groupby("year")["value"].transform(
        lambda s: s.rank(pct=True)
    )
    df["exceed_95"] = (df["value"] - 95).clip(lower=0)

    # 简单年内归一化（保持 0-1）
    def minmax(s):
        lo, hi = s.min(), s.max()
        return (s - lo) / (hi - lo) if hi > lo else 0.0

    df["_n_abs_pct_year"] = df.groupby("year")["abs_pct_year"].transform(minmax)
    df["_n_rel_dev"] = df.groupby("year")["rel_dev"].transform(minmax)
    df["_n_ex95"] = df.groupby("year")["exceed_95"].transform(minmax)

    df["HSI"] = (
        0.35 * df["_n_abs_pct_year"] + 0.35 * df["_n_rel_dev"] + 0.30 * df["_n_ex95"]
    )
    df.drop(columns=["_n_abs_pct_year", "_n_rel_dev", "_n_ex95"], inplace=True)

# 列名清洗&类型
df.columns = [c.strip() for c in df.columns]
for col in ["value", "rank", "mean_1901_2000", "HSI"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ====== 1. 全国均值趋势：HSI 与夏季最高温 ======
national = df.groupby("year", as_index=False).agg(
    HSI_mean=("HSI", "mean"), value_mean=("value", "mean")
)

plt.figure(figsize=(8, 5))
plt.plot(national["year"], national["HSI_mean"], linewidth=2)
plt.title("U.S. Average Heat Stress Index (HSI), 2000–2020")
plt.xlabel("Year")
plt.ylabel("HSI (avg)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "01_us_hsi_trend.png"), dpi=180)
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(national["year"], national["value_mean"], linewidth=2)
plt.title("U.S. Average Summer Tmax (°F), 2000–2020")
plt.xlabel("Year")
plt.ylabel("Tmax (°F)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_us_tmax_trend.png"), dpi=180)
plt.close()

# ====== 2. 各州 HSI 趋势（展示 Top 6 州）=====
state_mean = df.groupby(["state", "year"], as_index=False)["HSI"].mean()
state_pivot_last = state_mean[state_mean["year"].eq(state_mean["year"].max())]
top_states = (
    state_pivot_last.sort_values("HSI", ascending=False).head(6)["state"].tolist()
)

plt.figure(figsize=(9, 6))
for st in top_states:
    sub = state_mean[state_mean["state"] == st]
    plt.plot(sub["year"], sub["HSI"], label=st, linewidth=2)
plt.title("Top-6 States by HSI (latest year) – Trend 2000–2020")
plt.xlabel("Year")
plt.ylabel("HSI (state avg)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "03_states_top6_hsi_trends.png"), dpi=180)
plt.close()

# ====== 3. 县级：20 年平均 HSI Top/Bottom10（跨地区比较）=====
county_mean = df.groupby(["state", "name"], as_index=False)["HSI"].mean()
top10 = county_mean.sort_values("HSI", ascending=False).head(10)
bot10 = county_mean.sort_values("HSI", ascending=True).head(10)

# Top10
plt.figure(figsize=(9, 6))
labels = [f"{n}\n({s})" for s, n in zip(top10["state"], top10["name"])]
plt.barh(labels[::-1], top10["HSI"][::-1].values)
plt.title("Top 10 Counties by Average HSI (2000–2020)")
plt.xlabel("HSI (avg)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "04_top10_counties_hsi.png"), dpi=180)
plt.close()

# Bottom10
plt.figure(figsize=(9, 6))
labels = [f"{n}\n({s})" for s, n in zip(bot10["state"], bot10["name"])]
plt.barh(labels[::-1], bot10["HSI"][::-1].values)
plt.title("Bottom 10 Counties by Average HSI (2000–2020)")
plt.xlabel("HSI (avg)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "05_bottom10_counties_hsi.png"), dpi=180)
plt.close()

# ====== 4. 年度分布：按年 HSI 箱线图（看极端年）=====
# 为避免图过密，可抽样或直接箱线
years = sorted(df["year"].unique().tolist())
data = [df[df["year"] == y]["HSI"].dropna().values for y in years]

plt.figure(figsize=(10, 5))
plt.boxplot(data, labels=years, showfliers=False)
plt.title("Distribution of HSI by Year (2000–2020)")
plt.xlabel("Year")
plt.ylabel("HSI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "06_hsi_box_by_year.png"), dpi=180)
plt.close()

# ====== 5. 选定州（示例：Texas）内所有县的年均 HSI 趋势（可替换州名）=====
STATE_FOCUS = "Texas"  # 修改成你客户关注的州
st_df = df[df["state"] == STATE_FOCUS]
st_cnty = st_df.groupby(["name", "year"], as_index=False)["HSI"].mean()

# 画出该州所有县的淡线 + 州平均的粗线
plt.figure(figsize=(9, 6))
for nm, g in st_cnty.groupby("name"):
    plt.plot(g["year"], g["HSI"], linewidth=0.8, alpha=0.5)
st_avg = st_cnty.groupby("year", as_index=False)["HSI"].mean()
plt.plot(st_avg["year"], st_avg["HSI"], linewidth=3)
plt.title(f"{STATE_FOCUS}: County HSI Trends and State Average")
plt.xlabel("Year")
plt.ylabel("HSI")
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(OUT_DIR, f"07_{STATE_FOCUS.lower()}_county_trends.png"), dpi=180
)
plt.close()

print("All figures saved to:", OUT_DIR)

Loaded: /Users/liuliangcheng/Desktop/Duke/capstone/temperature/county_heat_index_2000_2020_with_HSI.csv


/var/folders/35/864rn6qs23n05rwx8x5t88700000gn/T/ipykernel_30111/4119986975.py:124: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=years, showfliers=False)


All figures saved to: /Users/liuliangcheng/Desktop/Duke/capstone/temperature/figs
